Reference: https://docs.databricks.com/aws/en/tables/features/deletion-vectors

In [0]:
%pip install git+https://github.com/jawong3/Databricks_Demos_and_Utilities.git@main
%restart_python

In [0]:
import time
from pyspark.sql import functions as f
from dbx_utils.benchmark import *

CATALOG = "data_ai_dbx_learning_workspace"
SCHEMA = "default"

ROWS = 25_000_000
NUM_FILES = 20
DELETE_MOD = 100

# Delta Cache will skew the benchmark results
spark.conf.set("spark.databricks.io.cache.enabled", "false")

RV_OFF = f"{CATALOG}.{SCHEMA}.deletion_vector_f"
RV_ON = f"{CATALOG}.{SCHEMA}.deletion_vector_t"

In [0]:
def create_table(full, dv_enabled):
    spark.sql(f"DROP TABLE IF EXISTS {full}")
    spark.sql(f"""
        CREATE TABLE {full} (id BIGINT, customer_id BIGINT, amount DOUBLE, payload STRING)
        USING DELTA
        TBLPROPERTIES ('delta.enableDeletionVectors' = '{str(dv_enabled).lower()}')
    """)
    (spark.range(ROWS)
        .withColumn("customer_id", (f.rand(1) * 100000).cast("long"))
        .withColumn("amount", f.rand(2) * 1000)
        .withColumn("payload", f.sha2(f.col("id").cast("string"), 256))
        .repartition(NUM_FILES)
        .write.mode("append").saveAsTable(full))
    
create_table(RV_OFF, dv_enabled=False)
create_table(RV_ON, dv_enabled=True)

In [0]:
def dv_prop(table):
    return (
        spark.sql(f"SHOW TBLPROPERTIES {table} ('delta.enableDeletionVectors')")
        .select(f.lit(table).alias("table"), f.col("value").alias("delta.enableDeletionVectors"))
    )

dv_prop(RV_OFF).unionByName(dv_prop(RV_ON)).show(truncate=False)

In [0]:
@time_runs
def read_table(full):
    return spark.table(full).agg(f.count("*"), f.sum("amount"), f.max("payload"))

# Shouldn't see any differences as its the same table with no modifications yet
print("RV_OFF Baseline")
df = read_table(RV_OFF)
print("RV_ON Baseline")
df = read_table(RV_ON)

In [0]:
@time_runs(runs=1)
def delete_mod(full, invert=False):
    pred = f"id % {DELETE_MOD} = 0"
    if invert: pred = f"NOT ({pred})"
    spark.sql(f"DELETE FROM {full} WHERE {pred}")

# Should notice a significant difference as without deletion vectors,
# (almost) all files had to be rewritten, whereas with deletion vectors,
# only metadata got appended, making the deletion operation far quicker
print("RV_OFF Minor Delete")
delete_mod(RV_OFF)
print("RV_ON Minor Delete")
delete_mod(RV_ON)

# Shouldn't notice the overhead of deletion vectors yet with a minor delete
print("RV_OFF Post Minor Delete Read")
df = read_table(RV_OFF)
print("RV_ON Post Minor Delete Read")
df = read_table(RV_ON)

In [0]:
create_table(RV_OFF, dv_enabled=False)
create_table(RV_ON, dv_enabled=True)

# Again should see a significant difference here
print("RV_OFF Major Delete")
delete_mod(RV_OFF, invert=True)
print("RV_ON Major Delete")
delete_mod(RV_ON, invert=True)

# Should notice the overhead of deletion vectors now as there is an abundance of metadata to scan over
print("RV_OFF Post Major Delete Read")
df = read_table(RV_OFF)
print("RV_ON Post Major Delete Read")
df = read_table(RV_ON)

In [0]:
# Resets the overhead by applying the metadata deletes and cleaing the deletion vectors
spark.sql(f"REORG TABLE {RV_ON} APPLY (PURGE)")

print("RV_ON Post Reorg Read")
df = read_table(RV_ON)

In [0]:
spark.sql(f"DROP TABLE IF EXISTS {RV_OFF}")
spark.sql(f"DROP TABLE IF EXISTS {RV_ON}")